In [1]:
# Cell 1: Load embedded dataset (Parquet recommended)
import pandas as pd
import numpy as np
import json

DATA_PATH = r"embeddings_out/dataset_with_embeddings.parquet"  # <-- change

df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.head()


(1114, 16)


,#,البيت,الشاعر,القصيدة,التعبير البلاغي,العنصر البلاغي المستعمل,العنصر البلاغي الثانوي,الثيمة المستعملة,الشرح,عدل/احذف,source_file,source_sheet,label_majaz,text_for_embedding,embedding,embedding_model
0,1,2,الكميت بن زيد الأسدي,شرح هاشمية 6، وافر، عَا,دَخِيلٌ يَهِيجُ سُقْمًا,مجاز,مجاز عقلي,بكاء الحسين وآله,None,NaN,تحليل بلاغي (10)__Sheet1.csv,Sheet1,1,دَخِيلٌ يَهِيجُ سُقْمًا,"[-0.04892054200172424, 0.008414781652390957, 0...",text-embedding-3-large
1,13,3,الكميت بن زيد الأسدي,شرح هاشمية 6، وافر، عَا,الضُّلُوعَا,مجاز,مجاز مرسل,بكاء الحسين وآله,المقصود ما جاور الضلوع من الحشا,NaN,تحليل بلاغي (10)__Sheet1.csv,Sheet1,1,الضُّلُوعَا,"[-0.026363924145698547, 0.0013875265140086412,...",text-embedding-3-large
2,16,4,الكميت بن زيد الأسدي,شرح هاشمية 6، وافر، عَا,يُرَقْرِقُ أسْجُمًا دِرَرًا وَسَكْبًا,مجاز,مجاز عقلي,بكاء الحسين وآله,"فاعل ""يرقرق"" (يصبّ) هو الهمّ، وهذا مجاز عقليّ.",NaN,تحليل بلاغي (10)__Sheet1.csv,Sheet1,1,يُرَقْرِقُ أسْجُمًا دِرَرًا وَسَكْبًا,"[0.0022120813373476267, 0.008848325349390507, ...",text-embedding-3-large
3,84,1,الكميت بن زيد الأسدي,شرح هاشمية 6، وافر، عَا,نَفَى الأَرَقُ الهُجُوعَا,مجاز,مجاز عقلي,بكاء الحسين وآله,ليست الأرق هو الفاعل الحقيقي الذي يبعد النوم ع...,NaN,تحليل بلاغي (10)__Sheet1.csv,Sheet1,1,نَفَى الأَرَقُ الهُجُوعَا,"[-0.015608849935233593, 0.0004104303661733866,...",text-embedding-3-large
4,9,7,النابغة الذبياني,قصيدة رقم 1، البسيط، دِ,القُتُودَ,مجاز,مجاز مرسل,الرحلة,None,NaN,تحليل بلاغي (11)__Sheet1.csv,Sheet1,1,القُتُودَ,"[-0.023851485922932625, -0.006078546866774559,...",text-embedding-3-large


In [2]:
# Cell 2: Prepare X (embeddings) and y (labels)
# embedding is stored as list[float] in parquet
if df["embedding"].dtype == object and isinstance(df["embedding"].iloc[0], str):
    # fallback if you loaded from CSV where embedding is JSON string
    df["embedding"] = df["embedding"].apply(json.loads)

X = np.vstack(df["embedding"].values).astype(np.float32)
y = df["label_majaz"].astype(int).values

print("X:", X.shape, "y:", y.shape)
print("Class balance:", {0: int((y==0).sum()), 1: int((y==1).sum())})


X: (1114, 3072) y: (1114,)
Class balance: {0: 557, 1: 557}


In [3]:
# Cell 3: Train/test split (70/30) + standardization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)


Train: (779, 3072) Test: (335, 3072)


In [4]:
# # Cell 4: Train MANY models on embeddings + evaluate (incl. neural nets)
# import numpy as np
# import pandas as pd

# from sklearn.linear_model import LogisticRegression, RidgeClassifier
# from sklearn.svm import LinearSVC, SVC
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.naive_bayes import GaussianNB
# from sklearn.ensemble import (
#     RandomForestClassifier, ExtraTreesClassifier,
#     AdaBoostClassifier, GradientBoostingClassifier
# )
# from sklearn.neural_network import MLPClassifier
# from sklearn.calibration import CalibratedClassifierCV
# from sklearn.metrics import (
#     accuracy_score, precision_recall_fscore_support,
#     roc_auc_score, confusion_matrix
# )

# def eval_model(name, clf, Xtr, ytr, Xte, yte):
#     clf.fit(Xtr, ytr)
#     y_pred = clf.predict(Xte)

#     # score for AUC if possible
#     y_score = None
#     if hasattr(clf, "predict_proba"):
#         y_score = clf.predict_proba(Xte)[:, 1]
#     elif hasattr(clf, "decision_function"):
#         y_score = clf.decision_function(Xte)

#     acc = accuracy_score(yte, y_pred)
#     prec, rec, f1, _ = precision_recall_fscore_support(yte, y_pred, average="binary", zero_division=0)
#     auc = roc_auc_score(yte, y_score) if y_score is not None else np.nan
#     cm = confusion_matrix(yte, y_pred)

#     return {
#         "model": name,
#         "accuracy": acc,
#         "precision": prec,
#         "recall": rec,
#         "f1": f1,
#         "roc_auc": auc,
#         "tn": cm[0,0], "fp": cm[0,1],
#         "fn": cm[1,0], "tp": cm[1,1],
#     }


# models = {}

# # 1) Strong linear baselines for embeddings
# models["LogReg_C1"]  = LogisticRegression(max_iter=8000, C=1.0, class_weight="balanced", random_state=42)
# models["LogReg_C3"]  = LogisticRegression(max_iter=8000, C=3.0, class_weight="balanced", random_state=42)
# models["LogReg_C10"] = LogisticRegression(max_iter=8000, C=10.0, class_weight="balanced", random_state=42)

# # RidgeClassifier is fast and often competitive
# models["Ridge"] = RidgeClassifier(class_weight="balanced", random_state=42)

# # Linear SVM (calibrated => probabilities + AUC)
# svm1 = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
# models["LinearSVM_C1(cal)"] = CalibratedClassifierCV(svm1, method="sigmoid", cv=5)

# svm3 = LinearSVC(C=3.0, class_weight="balanced", random_state=42)
# models["LinearSVM_C3(cal)"] = CalibratedClassifierCV(svm3, method="sigmoid", cv=5)

# # 2) Nonlinear options (sometimes help, sometimes not)
# # RBF SVC can be strong but slower; calibrate to get probabilities
# rbf = SVC(C=3.0, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=42)
# models["SVC_RBF_C3"] = rbf

# # KNN (sensitive to scaling; you already scaled)
# models["KNN_k5"]  = KNeighborsClassifier(n_neighbors=5)
# models["KNN_k15"] = KNeighborsClassifier(n_neighbors=15)

# # Naive Bayes on continuous features (sometimes decent baseline)
# models["GaussianNB"] = GaussianNB()

# # 3) Tree ensembles (often not best for dense embeddings, but worth testing)
# models["RandomForest_800"] = RandomForestClassifier(
#     n_estimators=800, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
# )
# models["ExtraTrees_1200"] = ExtraTreesClassifier(
#     n_estimators=1200, random_state=42, n_jobs=-1, class_weight="balanced_subsample"
# )
# models["GradBoost"] = GradientBoostingClassifier(random_state=42)
# models["AdaBoost"]  = AdaBoostClassifier(random_state=42)

# # 4) Neural net on embeddings (scikit MLP)
# # Try two sizes: small and medium. Early stopping helps.
# models["MLP_small"] = MLPClassifier(
#     hidden_layer_sizes=(256,),
#     activation="relu",
#     alpha=1e-4,
#     max_iter=200,
#     early_stopping=True,
#     random_state=42
# )
# models["MLP_medium"] = MLPClassifier(
#     hidden_layer_sizes=(512, 128),
#     activation="relu",
#     alpha=1e-4,
#     max_iter=250,
#     early_stopping=True,
#     random_state=42
# )

# results = []
# for name, clf in models.items():
#     # embeddings models generally benefit from scaling; use your scaled matrices
#     res = eval_model(name, clf, X_train_s, y_train, X_test_s, y_test)
#     results.append(res)

# results_df = pd.DataFrame(results).sort_values(by=["f1", "roc_auc"], ascending=False)
# results_df


In [5]:
import torch, platform
print("Platform:", platform.platform())
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA build:", torch.version.cuda)
print("Num GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    x = torch.randn(1000, 1000, device="cuda")
    print("Tensor device test:", x.device)
else:
    print("No GPU detected by PyTorch (likely CPU-only torch install or CUDA mismatch).")

Platform: Windows-10-10.0.26200-SP0
Torch version: 2.9.1+cpu
CUDA available: False
Torch CUDA build: None
Num GPUs: 0
No GPU detected by PyTorch (likely CPU-only torch install or CUDA mismatch).


In [6]:
# Cell 4 (TensorFlow, NO Trainer, NO Torch): Fine-tune HF models (Majaz vs Non-Majaz)

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, set_seed

set_seed(42)
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# -----------------------------
# 1) Pick text + label columns
# -----------------------------
TEXT_COL = "text_for_embedding"   # change if needed
LABEL_COL = "label_majaz"

df2 = df[[TEXT_COL, LABEL_COL]].copy()
df2[TEXT_COL] = df2[TEXT_COL].fillna("").astype(str).str.strip()
df2 = df2[df2[TEXT_COL].str.len() > 0].reset_index(drop=True)
df2[LABEL_COL] = df2[LABEL_COL].astype(int)

# -----------------------------
# 2) 70/30 split (stratified)
# -----------------------------
train_df, test_df = train_test_split(
    df2,
    test_size=0.30,
    random_state=42,
    stratify=df2[LABEL_COL]
)

# -----------------------------
# 3) tf.data builder
# -----------------------------
def make_tf_dataset(tokenizer, texts, labels, max_len=192, batch_size=16, shuffle=False):
    enc = tokenizer(
        list(texts),
        truncation=True,
        max_length=max_len,
        padding=True,
        return_tensors="tf",
    )

    y = tf.convert_to_tensor(np.array(labels, dtype=np.int32))
    ds = tf.data.Dataset.from_tensor_slices((dict(enc), y))

    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(texts), 10000), seed=42, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# -----------------------------
# 4) Eval metrics
# -----------------------------
def eval_tf_model(model, tokenizer, texts, labels, max_len=192, batch_size=32):
    ds = make_tf_dataset(tokenizer, texts, labels, max_len=max_len, batch_size=batch_size, shuffle=False)

    probs_all = []
    y_all = np.array(labels, dtype=np.int32)

    for xb, yb in ds:
        outputs = model(xb, training=False)
        logits = outputs.logits.numpy()
        # softmax to get prob of class 1
        exps = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs = exps / exps.sum(axis=1, keepdims=True)
        probs_all.append(probs[:, 1])

    p1 = np.concatenate(probs_all, axis=0)
    preds = (p1 >= 0.5).astype(int)

    acc = accuracy_score(y_all, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y_all, preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(y_all, p1)
    except ValueError:
        auc = float("nan")

    return {
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "roc_auc": float(auc),
    }

# -----------------------------
# 5) Train one model (Keras)
# -----------------------------
def finetune_one_tf(model_name, max_len=192, epochs=3, lr=2e-5, train_bs=16, eval_bs=32, out_dir="./tf_runs"):
    print("\n==============================")
    print("Training (TF):", model_name)
    print("==============================")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    train_ds = make_tf_dataset(
        tokenizer,
        train_df[TEXT_COL],
        train_df[LABEL_COL],
        max_len=max_len,
        batch_size=train_bs,
        shuffle=True
    )
    test_ds = make_tf_dataset(
        tokenizer,
        test_df[TEXT_COL],
        test_df[LABEL_COL],
        max_len=max_len,
        batch_size=eval_bs,
        shuffle=False
    )

    # TF model
    model = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(optimizer=optimizer, loss=loss)

    # best-by-F1 checkpoint (manual: evaluate each epoch and save)
    safe = model_name.replace("/", "_")
    run_dir = os.path.join(out_dir, safe)
    os.makedirs(run_dir, exist_ok=True)

    best_f1 = -1.0
    best_path = os.path.join(run_dir, "best_savedmodel")

    for ep in range(1, epochs + 1):
        model.fit(train_ds, epochs=1, verbose=1)
        metrics = eval_tf_model(model, tokenizer, test_df[TEXT_COL], test_df[LABEL_COL], max_len=max_len, batch_size=eval_bs)
        print(f"Epoch {ep} eval:", metrics)

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            # Save as SavedModel
            model.save_pretrained(best_path)
            tokenizer.save_pretrained(best_path)
            print("Saved new best:", best_path)

    # Final metrics = best_f1 seen
    final_metrics = eval_tf_model(model, tokenizer, test_df[TEXT_COL], test_df[LABEL_COL], max_len=max_len, batch_size=eval_bs)
    final_metrics["best_f1"] = float(best_f1)
    final_metrics["best_ckpt"] = best_path

    return final_metrics, run_dir

# -----------------------------
# 6) Run models
# -----------------------------
MODEL_LIST = [
    "aubmindlab/bert-base-arabertv2",
    "xlm-roberta-base",
]

all_results = []
run_dirs = {}

for m in MODEL_LIST:
    try:
        metrics, run_dir = finetune_one_tf(m, max_len=192, epochs=3, lr=2e-5, train_bs=16, eval_bs=32, out_dir="./tf_runs")
        all_results.append({"model": m, **metrics})
        run_dirs[m] = run_dir
    except Exception as e:
        print(f"[SKIP] {m} failed: {e}")

results_df = pd.DataFrame(all_results)
if not results_df.empty:
    results_df = results_df.sort_values(by=["f1", "roc_auc"], ascending=False).reset_index(drop=True)

results_df



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  Fil

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  Fil

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  Fil

...


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\USER\.conda\envs\majaz-tf\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  Fil

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [ ]:
# Cell 4b (Optional): PyTorch MLP on embeddings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

Xtr = torch.tensor(X_train_s, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

Xte = torch.tensor(X_test_s, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(Xte, yte), batch_size=256, shuffle=False)

class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 512),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MLP(X_train_s.shape[1]).to(device)

# If data is perfectly balanced, pos_weight=1 is fine.
# If not balanced, set pos_weight = (#neg/#pos)
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
pos_weight = torch.tensor([neg / max(pos, 1)], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def predict_scores(m, X):
    m.eval()
    with torch.no_grad():
        logits = m(X.to(device)).cpu().numpy().reshape(-1)
        probs = 1 / (1 + np.exp(-logits))
    return probs

best_f1 = -1
best_state = None

for epoch in range(1, 21):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        opt.step()

    probs = predict_scores(model, Xte)
    pred = (probs >= 0.5).astype(int)

    acc = accuracy_score(y_test, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred, average="binary", zero_division=0)
    auc = roc_auc_score(y_test, probs)

    print(f"epoch {epoch:02d} | acc={acc:.4f} f1={f1:.4f} prec={prec:.4f} rec={rec:.4f} auc={auc:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# Load best epoch
if best_state is not None:
    model.load_state_dict(best_state)

probs = predict_scores(model, Xte)
pred = (probs >= 0.5).astype(int)
acc = accuracy_score(y_test, pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred, average="binary", zero_division=0)
auc = roc_auc_score(y_test, probs)

print("\nBEST PyTorch MLP:")
print({"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc})


In [ ]:
# Cell 5: Show the best model's confusion matrix + a few errors to inspect
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

best_name = results_df.iloc[0]["model"]
best_clf = models[best_name]

# Refit best model (already fit above, but this ensures it's in sync)
best_clf.fit(X_train_s, y_train)
y_pred_best = best_clf.predict(X_test_s)

disp = ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best)
plt.show()

# Inspect some mistakes
test_df = df.iloc[[i for i in range(len(df)) if True]]  # placeholder to keep df reference
# Rebuild a test index mapping from the split:
# We'll re-run split with indices to retrieve rows.
from sklearn.model_selection import train_test_split

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(idx, test_size=0.30, random_state=42, stratify=y)

err_idx = idx_test[y_pred_best != y_test]
print("Num errors:", len(err_idx))

cols_to_show = [c for c in ["الشاعر", "القصيدة", "التعبير البلاغي", "العنصر البلاغي المستعمل", "label_majaz"] if c in df.columns]
df.loc[err_idx, cols_to_show].head(20)


In [ ]:
# Cell 6: Save the BEST sklearn model + scaler (based on results_df)
import joblib
from pathlib import Path

OUT_DIR = Path(r"embeddings_out\models")  # <-- change if needed
OUT_DIR.mkdir(parents=True, exist_ok=True)

best_name = results_df.iloc[0]["model"]
best_clf = models[best_name]
best_clf.fit(X_train_s, y_train)

joblib.dump(
    {"model": best_clf, "scaler": scaler, "model_name": best_name},
    OUT_DIR / "majaz_classifier.joblib"
)

print("Best model:", best_name)
print("Saved to:", OUT_DIR / "majaz_classifier.joblib")
